# CO2Dot multi-device calibration

Detect **every** CO2Dot plugged into this PC over USB and read a `spec_flash`
(dark / lit / difference spectra) from each one.

`CO2Dot.connect()` stops at the *first* device it finds, so for a multi-device
bench we scan every serial port ourselves and keep every connection that answers
`hello` as a CO2Dot. Each port is opened **once and kept open** — opening a
USB-CDC port resets the MCU, so we never open/close/re-open.

Prereqs: run from this folder so `import co2dot` resolves, and all CO2Dots plugged in.

In [1]:
from co2dot import CO2Dot, list_ports
import json
from datetime import datetime
from pathlib import Path
import time


def connect_all(hello_timeout_s=4.0):
    """Open every CO2Dot on the PC and return a list of connected handles.

    Scans all serial ports; for each, uses CO2Dot.connect(port=...), which opens
    the port once, waits out the USB-CDC reset + firmware boot, and keeps the
    connection if the device answers `hello` as a CO2Dot. Ports that are not a
    CO2Dot (or are busy) are skipped. Connections stay open — close them with
    the cleanup cell at the bottom when done.
    """
    dots = []
    for port in list_ports():
        try:
            dot = CO2Dot.connect(port=port, hello_timeout_s=hello_timeout_s)
        except Exception as e:
            print(f"  {port}: skipped ({e})")
            continue
        print(f"  found {dot.device} on {dot.port}")
        dots.append(dot)
    return dots


print("scanning serial ports ...")
dots = connect_all()
print(f"\n{len(dots)} CO2Dot(s) connected: {[d.port for d in dots]}")

scanning serial ports ...
  found CO2Dot on COM8
  COM16: skipped (No CO2Dot found. Last: COM16: opened but no CO2Dot hello)
  found CO2Dot on COM19
  found CO2Dot on COM20

3 CO2Dot(s) connected: ['COM8', 'COM19', 'COM20']


In [3]:
for idx, dot in enumerate(dots):
    dot.set_gain(idx+2)  # reset gain to 1 for all devices

In [4]:
for dot in dots:
    print(dot.status())  

{'spectrometer': {'model': 'AS7341', 'available': True, 'atime': 100, 'astep': 999, 'gain': 2, 'led': 0}, 'bme': {'available': True}}
{'spectrometer': {'model': 'AS7341', 'available': True, 'atime': 100, 'astep': 999, 'gain': 3, 'led': 0}, 'bme': {'available': True}}
{'spectrometer': {'model': 'AS7341', 'available': True, 'atime': 100, 'astep': 999, 'gain': 4, 'led': 0}, 'bme': {'available': True}}


In [ ]:
REPEAT = 10
DELAY = 0.5
LED_CURRENT = 1

results = {}

for _ in range(REPEAT):
    for dot in dots:
        try:
            flash = dot.spec_flash(LED_CURRENT)     # sends `spec_flash,1`; reads dark / lit / diff
            results[dot.port] = flash
            print(f"{dot.port} [{dot.device}] {flash['model']}: "
                f"{len(flash['diff'])} channels  diff={flash['diff']}")
        except Exception as e:
            print(f"{dot.port}: spec_flash FAILED ({e})")
    time.sleep(DELAY)

COM8 [CO2Dot] AS7341: 10 channels  diff={'f1_415': 36, 'f2_445': 101, 'f3_480': 69, 'f4_515': 168, 'f5_555': 336, 'f6_590': 306, 'f7_630': 306, 'f8_680': 498, 'clear': 436, 'nir': 42}
COM19 [CO2Dot] AS7341: 10 channels  diff={'f1_415': 73, 'f2_445': 205, 'f3_480': 123, 'f4_515': 299, 'f5_555': 597, 'f6_590': 552, 'f7_630': 568, 'f8_680': 924, 'clear': 736, 'nir': 71}
COM20 [CO2Dot] AS7341: 10 channels  diff={'f1_415': 137, 'f2_445': 345, 'f3_480': 252, 'f4_515': 602, 'f5_555': 1167, 'f6_590': 1107, 'f7_630': 1113, 'f8_680': 1799, 'clear': 1441, 'nir': 146}
COM8 [CO2Dot] AS7341: 10 channels  diff={'f1_415': 36, 'f2_445': 101, 'f3_480': 69, 'f4_515': 168, 'f5_555': 336, 'f6_590': 306, 'f7_630': 305, 'f8_680': 498, 'clear': 436, 'nir': 42}
COM19 [CO2Dot] AS7341: 10 channels  diff={'f1_415': 73, 'f2_445': 205, 'f3_480': 123, 'f4_515': 299, 'f5_555': 597, 'f6_590': 552, 'f7_630': 568, 'f8_680': 923, 'clear': 736, 'nir': 71}
COM20 [CO2Dot] AS7341: 10 channels  diff={'f1_415': 137, 'f2_445': 

In [6]:
# Save all devices' spectra to one timestamped JSON (keyed by serial port).
outdir = Path("co2dot_data")
outdir.mkdir(parents=True, exist_ok=True)
record = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "led_current": LED_CURRENT,
    "devices": results,
}
path = outdir / (datetime.now().strftime("%Y-%m-%d_%H-%M-%S") + "_multidevice_specflash.json")
path.write_text(json.dumps(record, indent=2), encoding="utf-8")
print("saved", path)

saved co2dot_data\2026-07-14_17-03-04_multidevice_specflash.json


In [7]:
# # Close every open connection when finished (frees the serial ports).
# for dot in dots:
#     dot.close()
# print(f"closed {len(dots)} connection(s)")